# 1. Propósito

Resolver exclusivamente la composición ordenada $R_1U_1$, conservar su representación factorada y tratar por separado el semiproducto $R_1\widehat G_1$. No se estudia el factor Wiener--Hopf ni se afirma Fredholmness.

In [1]:
import sympy as sp
from symbolic_operator_calculus import (
    ClosureObligationStatus, DilationFrequencyCase, MinimalLemmaChoice,
    Product, R1, U1, StandardMellinMembership,
    build_coefficient_semiproduct_trace, build_damping_variation_lemma,
    build_first_pivot_closure_analysis, build_oscillatory_mellin_factor,
    build_right_dilation_composition_trace,
    render_coefficient_semiproduct_latex,
    render_minimal_closure_decision_yaml,
    render_right_dilation_composition_latex,
)
lam, gamma = sp.symbols('lambda gamma')
trace = build_right_dilation_composition_trace(lam, gamma)
assert trace.ast_product == Product((R1, U1))
print('Phase Q: exact right-dilation composition; R1 remains atomic')

Phase Q: exact right-dilation composition; R1 remains atomic


# 2. Convenciones

Se usa $d\mu(x)=dx/x$, $\mathcal Mf(\lambda)=\int_0^\infty f(x)x^{-i\lambda}dx/x$ y la inversa con $x^{i\lambda}$.

In [2]:
print('Mellin transform: x**(-I*lambda); inverse: x**(I*lambda)')
print('Measure: dmu(x)=dx/x; gamma>0')

Mellin transform: x**(-I*lambda); inverse: x**(I*lambda)
Measure: dmu(x)=dx/x; gamma>0


# 3. Definición de $\operatorname{Op}(a)$

La cuantización Mellin de Kohn--Nirenberg mantiene la variable espacial en el factor izquierdo del símbolo.

In [3]:
print('[Op(a)f](x)=(1/2pi) int_R int_R+ a(x,lambda)(x/y)**(I*lambda) f(y) dy/y dlambda')
print(trace.exact_identity.evidence.quantization)

[Op(a)f](x)=(1/2pi) int_R int_R+ a(x,lambda)(x/y)**(I*lambda) f(y) dy/y dlambda
paper equation: Mellin Kohn--Nirenberg kernel with (x/y)^(i lambda)


# 4. Definición de $U_\gamma$

La normalización ponderada desaparece exactamente después de conjugar por $\Phi_\delta$.

In [4]:
x = sp.Symbol('x', positive=True)
gamma_pos = sp.Symbol('gamma', positive=True)
kappa = sp.Symbol('kappa', real=True)
normalization = sp.powsimp(x**kappa * gamma_pos**kappa * (gamma_pos*x)**(-kappa), force=True)
assert normalization == 1
print(trace.exact_identity.evidence.normalized_action)
print('residual scalar factor:', normalization)

Phi_delta U_gamma Phi_delta^(-1)g(x)=g(gamma x)
residual scalar factor: 1


# 5. Derivación exacta

En una clase densa y con truncación simétrica en frecuencia, la sustitución $z=\gamma y$ se hace dentro de la integral interior; no se intercambian integrales ni se conmuta ningún factor.

In [5]:
assert trace.exact_identity.left == Product((R1, U1))
print(render_right_dilation_composition_latex(trace))
print(trace.exact_identity.evidence.substitution)
print(trace.exact_identity.evidence.continuity_extension)

R_{1}\,U_{1}\overset{\mathrm{exact}}{=}\Phi_\delta^{-1}\operatorname{Op}_{\mathrm{right}\text{-}\gamma_1}\!\left(r_1,d_{\gamma_1}\right)\Phi_\delta\quad[d_{\gamma_1}(\lambda)=\gamma_1^{i\lambda};\ r_1d_{\gamma_1}\in\widetilde{\mathcal E}:\ \text{not demonstrated}]
z=gamma y in the inner integral; integrations are not interchanged
boundedness of Op(r1) and isometry of the dilation give the unique extension


# 6. Prueba del signo

La sustitución en la transformada y la función log-gaussiana seleccionan el signo positivo.

In [6]:
d_gamma = trace.factorized_symbol.oscillatory_factor.expression
assert d_gamma == sp.Pow(gamma, sp.I*lam, evaluate=False)
gaussian_transform = sp.sqrt(sp.pi) * sp.exp(-lam**2/4)
print(d_gamma)
print('log-Gaussian shifted transform:', d_gamma * gaussian_transform)

gamma**(I*lambda)
log-Gaussian shifted transform: sqrt(pi)*gamma**(I*lambda)*exp(-lambda**2/4)


# 7. Clasificación de $d_\gamma$

Si $\gamma\ne1$, su derivada tiene módulo constante $|\log\gamma|$, la variación total es infinita y no se certifica membresía en $V(\mathbb R)$.

In [7]:
osc = trace.factorized_symbol.oscillatory_factor
assert osc.standard_v_membership is StandardMellinMembership.EXCLUDED_NONTRIVIAL_DILATION
print('membership:', osc.standard_v_membership.value)
print('proof:', osc.proof[0])

membership: EXCLUDED_NONTRIVIAL_DILATION
proof: for gamma>0, gamma!=1, |d_gamma'|=|log gamma| and total variation is infinite


# 8. Representación factorada

El par ordenado $(r_1,d_{\gamma_1})$ evita declarar falsamente que el producto puntual pertenece a la clase estándar.

In [8]:
factorized = trace.factorized_symbol
assert factorized.standard_product_membership is StandardMellinMembership.NOT_DEMONSTRATED
print('provisional class:', factorized.provisional_class)
print('standard membership:', factorized.standard_product_membership.value)
print('open properties:', [item.key for item in factorized.pending_properties])

provisional class: tilde-E^(right-gamma)
standard membership: NOT_DEMONSTRATED
open properties: ['right_gamma_sums', 'right_gamma_products', 'right_gamma_left_composition', 'right_gamma_involution', 'right_gamma_fredholm_symbol', 'right_gamma_norm_closure', 'right_gamma_wiener_hopf']


# 9. Caso $\gamma=1$

La frecuencia nula se mantiene separada del caso no trivial.

In [9]:
identity_factor = build_oscillatory_mellin_factor(lam, gamma, case=DilationFrequencyCase.IDENTITY)
assert identity_factor.expression == 1
assert identity_factor.standard_v_membership is StandardMellinMembership.CERTIFIED
print('d_1 =', identity_factor.expression, '; membership:', identity_factor.standard_v_membership.value)

d_1 = 1 ; membership: CERTIFIED


# 10. Caso $R_1\widehat G_1$

Este caso usa el Teorema 3.3 de KKL 2014 y conserva fuerza módulo compactos; no se mezcla con la identidad de dilatación.

In [10]:
coefficient = build_coefficient_semiproduct_trace()
print(render_coefficient_semiproduct_latex(coefficient))
print('status:', coefficient.rule.certification_status.value)
print('source:', coefficient.relation.evidence.theorem)

\operatorname{Op}(r_1)M_{\widehat G_1}\simeq\operatorname{Op}(r_1\widehat G_1)\quad\bmod\mathcal K\quad[\mathrm{CERTIFIED\_MOD\_COMPACT}]
status: CERTIFIED_MOD_COMPACT
source: KKL2014TwoShifts Theorem 3.3


# 11. Lema de amortiguamiento

Para $h\in V\cap L^1$, la oscilación puede absorberse con una cota explícita. No se sustituye $L^1$ por $C_0$.

In [11]:
damping = build_damping_variation_lemma(gamma_pos)
print('status:', damping.status.value)
print('variation bound:', damping.variation_bound)
print(damping.conclusion)

status: ANALYTICALLY_PROVED
variation bound: L1_h*Abs(log(gamma)) + Var_h
Var(d_gamma h) <= Var(h)+|log gamma| ||h||_1 and zero endpoint limits are preserved


# 12. Obligaciones restantes

P-02 y P-03 se cierran a fuerzas diferentes. P-04 continúa bloqueada porque falta un marco admisible uniforme.

In [12]:
closure = build_first_pivot_closure_analysis()
phase_q_nodes = {item.identifier: item.status.value for item in closure.obligations[:4]}
assert phase_q_nodes['P-02'] == ClosureObligationStatus.ANALYTICALLY_PROVED.value
assert phase_q_nodes['P-03'] == ClosureObligationStatus.ANALYTICALLY_PROVED.value
assert phase_q_nodes['P-04'] == ClosureObligationStatus.BLOCKED.value
print(phase_q_nodes)

{'P-01': 'BLOCKED', 'P-02': 'ANALYTICALLY_PROVED', 'P-03': 'ANALYTICALLY_PROVED', 'P-04': 'BLOCKED'}


# 13. Impacto sobre H1/H2/H3

La fórmula corta de dilatación ya es exacta y el caso coeficiente ya está controlado módulo compactos. Aún no existe una clase uniforme para H1; H2/H3 permanecen bloqueados y la decisión sigue siendo `NONE`.

In [13]:
assert closure.decision.decision is MinimalLemmaChoice.NONE
print(render_minimal_closure_decision_yaml(closure.decision))
print('candidate statuses:', {item.identifier.value: item.status.value for item in closure.candidates})

decision: NONE
confidence: high
blocking_obligations:
  - "P-01"
  - "P-04"
evidence:
  - "phase-q:exact-right-dilation-definition-proof"
  - "phase-q:certified-Ghat1-semiproduct"
  - "paper:normalized-wh-blocks-section-six"
  - "KKL2014TwoShifts:3.3"
  - "KKL2014TwoShifts:4.4"
  - "Karlovich2025Cusps:3.3"
rationale: "The two H1 cases now have separate certified relations, but the exact factorized U1 output has no proved closed admissible calculus. H2 and H3 also require an unproved identification of the localized Wplus_12."
prerequisite_statement: "Prove the minimum closure properties needed for the factorized pair (r1,d_gamma1), without forcing it into E-tilde; independently identify Wplus_12 before attempting H2."

candidate statuses: {'H1': 'BLOCKED', 'H2': 'BLOCKED', 'H3': 'BLOCKED'}
